# Dense–Sparse TT decomposition with superweights and LoRA

In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=7

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'src').exists() else CWD.parent
ARTIFACT_ROOT = CWD if (CWD / 'dense_sparse_tt_helpers.py').exists() else (CWD / '..').resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ARTIFACT_ROOT) not in sys.path:
    sys.path.insert(0, str(ARTIFACT_ROOT))

print('Repo root:', REPO_ROOT)
print('Artifact root:', ARTIFACT_ROOT)
print('Has src:', (REPO_ROOT / 'src').exists())
print('Has dense_sparse_tt_helpers.py:', (ARTIFACT_ROOT / 'dense_sparse_tt_helpers.py').exists())

In [ ]:
import gc
import json
import math

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.tt_llm import (
    TTLinear,
    cleanup_memory,
    factor_int_balanced,
    format_cuda_memory,
    get_module_by_name,
    infer_input_device,
    replace_llama_ffn_with_tt,
    replace_tt_with_dense_reconstruction,
    set_module_by_name,
)
from dense_sparse_tt_helpers import dense_sparse_tt_from_linear_tntorch_sum, dense_sparse_tt_from_linear_exact_sparse

try:
    from src.utils.eval import eval_ppl
except Exception:
    eval_ppl = None

pd.set_option('display.max_colwidth', 220)
torch.set_grad_enabled(True)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Configuration

In [ ]:
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'
TARGET_LAYER = 1
TARGET_MODULE = 'mlp.down_proj'
TARGET_MODULE_NAME = f'model.layers.{TARGET_LAYER}.{TARGET_MODULE}'
TARGET_SUPERWEIGHT_COORDS = [(2533, 7890)]

ORDER = 12
TT_RANK = 500
OUTLIER_FRACTIONS = [5e-7, 1e-6, 5e-6]
TOKEN_CHUNK_SIZE = 128
DECOMPOSE_DEVICE = 'cpu'
DECOMPOSE_DTYPE = torch.float64
MODEL_DTYPE = torch.float16

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
MAX_STEPS = 100
GRAD_ACCUM_STEPS = 8
TRAIN_BATCH_SIZE = 1
TRAIN_SEQ_LEN = 256
NUM_TRAIN_SEQUENCES = 512

RUN_PPL = True
PPL_SEQLEN = 4096

OUTPUT_JSON = REPO_ROOT / 'dense_sparse_tt_superweight_experiments.json'

In [ ]:
def clean_memory_local(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=MODEL_DTYPE,
        device_map='auto',
        low_cpu_mem_usage=True,
    )
    model.eval()
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    return model, tokenizer


def build_wikitext2_train_loader(
    tokenizer,
    *,
    seq_len: int,
    batch_size: int,
    num_sequences: int,
    seed: int = 0,
):
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    text = '\n\n'.join(ds['text'])
    encoded = tokenizer(text, return_tensors='pt')
    ids = encoded.input_ids[0]

    n_blocks = ids.numel() // seq_len
    ids = ids[: n_blocks * seq_len].view(n_blocks, seq_len)

    gen = torch.Generator().manual_seed(seed)
    if ids.shape[0] > num_sequences:
        perm = torch.randperm(ids.shape[0], generator=gen)[:num_sequences]
        ids = ids[perm]

    dataset = TensorDataset(ids)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


def evaluate_wikitext2_ppl(model, tokenizer, *, seqlen: int):
    if eval_ppl is None:
        raise RuntimeError('src.utils.eval.eval_ppl is not available in this repo layout')
    ppl = eval_ppl(model, tokenizer, datasets=['wikitext2'], seqlen=seqlen)
    return float(ppl['wikitext2'])


def get_target_linear(model):
    module = get_module_by_name(model, TARGET_MODULE_NAME)
    return module


@torch.no_grad()
def get_effective_target_weight(model):
    module = get_target_linear(model)
    if isinstance(module, nn.Linear):
        return module.weight.detach().float().cpu().contiguous()
    if isinstance(module, TTLinear):
        return module.to_dense_weight().detach().float().cpu().contiguous()
    if isinstance(module, LoRAOnFrozenLinear):
        return module.merged_linear().weight.detach().float().cpu().contiguous()
    raise TypeError(f'Unsupported target module type: {type(module).__name__}')


def summarize_superweight_error(current_weight: torch.Tensor, baseline_weight: torch.Tensor, coords):
    rows = []
    for row, col in coords:
        orig = float(baseline_weight[row, col].item())
        cur = float(current_weight[row, col].item())
        abs_error = abs(cur - orig)
        rel_error = abs_error / max(abs(orig), 1e-12)
        rows.append({
            'row': int(row),
            'col': int(col),
            'original_value': orig,
            'current_value': cur,
            'abs_error': abs_error,
            'rel_error': rel_error,
        })
    df = pd.DataFrame(rows)
    return {
        'superweight_abs_error_mean': float(df['abs_error'].mean()),
        'superweight_abs_error_max': float(df['abs_error'].max()),
        'superweight_rel_error_mean': float(df['rel_error'].mean()),
        'superweight_rel_error_max': float(df['rel_error'].max()),
        'superweight_error_detail': rows,
    }


def summarize_layer_error(current_weight: torch.Tensor, baseline_weight: torch.Tensor):
    diff = current_weight - baseline_weight
    fro_rel = float((torch.linalg.norm(diff) / torch.linalg.norm(baseline_weight).clamp_min(1e-12)).item())
    max_abs = float(diff.abs().max().item())
    return {
        'target_layer_fro_rel_error': fro_rel,
        'target_layer_max_abs_error': max_abs,
    }


@torch.no_grad()
def stage_row(experiment: str, method: str, stage: str, model, tokenizer, baseline_target_weight: torch.Tensor, *, outlier_fraction=None, method_stats=None):
    ppl = evaluate_wikitext2_ppl(model, tokenizer, seqlen=PPL_SEQLEN) if RUN_PPL else float('nan')
    module = get_target_linear(model)
    current_weight = get_effective_target_weight(model)
    sw_stats = summarize_superweight_error(current_weight, baseline_target_weight, TARGET_SUPERWEIGHT_COORDS)
    layer_stats = summarize_layer_error(current_weight, baseline_target_weight)
    row = {
        'experiment': experiment,
        'method': method,
        'outlier_fraction': outlier_fraction,
        'stage': stage,
        'module_type': type(module).__name__,
        'wikitext2_ppl': ppl,
        **sw_stats,
        **layer_stats,
    }
    if method_stats is not None:
        row.update(method_stats)
    return row

In [ ]:
def decompose_target_layer_regular_inplace(model, *, tt_rank: int):
    summaries = replace_llama_ffn_with_tt(
        model,
        layer_indices=[TARGET_LAYER],
        tt_rank=tt_rank,
        order=ORDER,
        projections=('down_proj',),
        decompose_dtype=DECOMPOSE_DTYPE,
        decompose_device=DECOMPOSE_DEVICE,
        token_chunk_size=TOKEN_CHUNK_SIZE,
    )
    tt_module = get_target_linear(model)
    tt_ranks = list(tt_module.tt_ranks)
    return {
        'method_compression_ratio': float(tt_module.compression_ratio()),
        'tt_ranks': tt_ranks,
        'combined_tt_max_rank': max(tt_ranks[1:-1]) if len(tt_ranks) > 2 else 1,
        'inlier_tt_ranks': tt_ranks,
        'inlier_tt_max_rank': max(tt_ranks[1:-1]) if len(tt_ranks) > 2 else 1,
        'sparse_tt_ranks': None,
        'sparse_tt_max_rank': None,
        'kept_outlier_count': 0,
        'outlier_count_exact': 0,
        'kept_fraction_actual': 0.0,
    }


@torch.no_grad()
def decompose_target_layer_dense_sparse_inplace(model, *, tt_rank: int, outlier_fraction: float):
    dense_module = get_target_linear(model)
    if not isinstance(dense_module, nn.Linear):
        raise TypeError(f'Expected nn.Linear before dense-sparse decomposition, got {type(dense_module).__name__}')

    tt_module, summary = dense_sparse_tt_from_linear_exact_sparse(
        dense_module,
        tt_rank_inliers=tt_rank,
        outlier_fraction=outlier_fraction,
        include_coords=TARGET_SUPERWEIGHT_COORDS,
        order=ORDER,
        decompose_dtype=DECOMPOSE_DTYPE,
        decompose_device=DECOMPOSE_DEVICE,
        output_device=dense_module.weight.device,
        token_chunk_size=TOKEN_CHUNK_SIZE,
    )
    set_module_by_name(model, TARGET_MODULE_NAME, tt_module)
    combined_tt_ranks = list(summary.combined_tt_ranks)
    inlier_tt_ranks = list(summary.inlier_tt_ranks)
    sparse_tt_ranks = list(summary.sparse_tt_ranks)
    return {
        'method_compression_ratio': summary.compression_ratio,
        'tt_ranks': combined_tt_ranks,
        'combined_tt_max_rank': max(combined_tt_ranks[1:-1]) if len(combined_tt_ranks) > 2 else 1,
        'inlier_tt_ranks': inlier_tt_ranks,
        'inlier_tt_max_rank': max(inlier_tt_ranks[1:-1]) if len(inlier_tt_ranks) > 2 else 1,
        'sparse_tt_ranks': sparse_tt_ranks,
        'sparse_tt_max_rank': max(sparse_tt_ranks[1:-1]) if len(sparse_tt_ranks) > 2 else 1,
        'kept_outlier_count': summary.kept_outlier_count,
        'outlier_count_exact': summary.kept_outlier_count,
        'kept_fraction_actual': summary.kept_fraction_actual,
    }


@torch.no_grad()
def reconstruct_target_layer_inplace(model):
    module = get_target_linear(model)
    if not isinstance(module, TTLinear):
        raise TypeError(f'Expected TTLinear, got {type(module).__name__}')
    set_module_by_name(model, TARGET_MODULE_NAME, module.to_linear())



def count_trainable_parameters(model) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def freeze_all_parameters(model):
    for p in model.parameters():
        p.requires_grad = False

In [ ]:
class LoRAOnFrozenLinear(nn.Module):
    def __init__(self, base_module: nn.Linear, r: int = 16, alpha: int = 32, dropout: float = 0.0):
        super().__init__()
        if not isinstance(base_module, nn.Linear):
            raise TypeError(f'Expected nn.Linear, got {type(base_module).__name__}')

        self.base_module = base_module
        self.in_features = int(base_module.in_features)
        self.out_features = int(base_module.out_features)
        self.r = int(r)
        self.alpha = int(alpha)
        self.scaling = float(alpha) / float(r)
        self.dropout = nn.Dropout(dropout)

        for p in self.base_module.parameters():
            p.requires_grad = False

        base_param = next(base_module.parameters())
        base_device = base_param.device
        self.lora_A = nn.Parameter(torch.empty(self.r, self.in_features, device=base_device, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, self.r, device=base_device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.base_module(x)
        x_work = self.dropout(x)
        x_lora = x_work.to(dtype=self.lora_A.dtype, device=self.lora_A.device)
        lora = (x_lora @ self.lora_A.t()) @ self.lora_B.t()
        lora = lora * self.scaling
        return base + lora.to(device=base.device, dtype=base.dtype)

    @torch.no_grad()
    def merged_linear(self) -> nn.Linear:
        base_weight = self.base_module.weight.detach()
        base_bias = None if self.base_module.bias is None else self.base_module.bias.detach().clone()
        device = self.base_module.weight.device
        dtype = self.base_module.weight.dtype

        delta = (self.lora_B @ self.lora_A) * self.scaling
        merged = nn.Linear(
            self.in_features,
            self.out_features,
            bias=base_bias is not None,
            device=device,
            dtype=dtype,
        )
        merged.weight.copy_(base_weight + delta.to(device=device, dtype=dtype))
        if base_bias is not None:
            merged.bias.copy_(base_bias.to(device=device, dtype=dtype))
        return merged



def attach_lora_to_target_module(model, *, r: int, alpha: int, dropout: float):
    base = get_target_linear(model)
    if not isinstance(base, nn.Linear):
        raise TypeError(f'LoRA attachment here expects reconstructed nn.Linear, got {type(base).__name__}')
    wrapper = LoRAOnFrozenLinear(base, r=r, alpha=alpha, dropout=dropout)
    set_module_by_name(model, TARGET_MODULE_NAME, wrapper)


@torch.no_grad()
def merge_target_lora_to_dense_inplace(model):
    module = get_target_linear(model)
    if not isinstance(module, LoRAOnFrozenLinear):
        raise TypeError(f'Expected LoRAOnFrozenLinear, got {type(module).__name__}')
    set_module_by_name(model, TARGET_MODULE_NAME, module.merged_linear())


In [ ]:
def run_lora_training(
    model,
    tokenizer,
    *,
    max_steps: int,
    grad_accum_steps: int,
    batch_size: int,
    seq_len: int,
    num_sequences: int,
    lr: float,
    weight_decay: float,
    seed: int = 0,
):
    freeze_all_parameters(model)
    for name, p in model.named_parameters():
        if 'lora_' in name:
            p.requires_grad = True

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    loader = build_wikitext2_train_loader(
        tokenizer,
        seq_len=seq_len,
        batch_size=batch_size,
        num_sequences=num_sequences,
        seed=seed,
    )

    model.train()
    history = []
    device = infer_input_device(model)

    optimizer.zero_grad(set_to_none=True)
    step_count = 0
    tokens_seen = 0
    micro_step = 0
    pbar = tqdm(total=max_steps, desc='LoRA training', leave=True)
    loader_iter = iter(loader)

    while step_count < max_steps:
        try:
            (input_ids,) = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            (input_ids,) = next(loader_iter)

        micro_step += 1
        input_ids = input_ids.to(device)
        tokens_seen += int(input_ids.numel())

        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss / grad_accum_steps
        loss.backward()

        if micro_step % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            step_count += 1
            current_loss = float(outputs.loss.detach().float().cpu().item())
            history.append({'step': step_count, 'tokens_seen': tokens_seen, 'loss': current_loss})
            pbar.update(1)
            pbar.set_postfix(loss=f'{current_loss:.4f}', tokens=tokens_seen)

    pbar.close()
    model.eval()
    return pd.DataFrame(history)

## Sanity check


In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print(format_cuda_memory())
print('Target module:', TARGET_MODULE_NAME)
print('Target module type:', type(get_target_linear(model)).__name__)
print('Target module shape:', get_target_linear(model).out_features, 'x', get_target_linear(model).in_features)
baseline_target_weight = get_effective_target_weight(model)
print('Baseline superweight values:', [float(baseline_target_weight[r, c]) for r, c in TARGET_SUPERWEIGHT_COORDS])
clean_memory_local(model, tokenizer)

## Experiment A: baseline vs regular TT vs dense-sparse TT


In [ ]:
def experiment_no_lora_methods():
    rows = []

    base_model, base_tokenizer = load_model_and_tokenizer(MODEL_NAME)
    baseline_target_weight = get_effective_target_weight(base_model)
    rows.append(stage_row('expA_no_lora', 'baseline', 'baseline', base_model, base_tokenizer, baseline_target_weight))
    clean_memory_local(base_model, base_tokenizer)

    # regular TT
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    method_stats = decompose_target_layer_regular_inplace(model, tt_rank=TT_RANK)
    reconstruct_target_layer_inplace(model)
    rows.append(
        stage_row(
            'expA_no_lora',
            'regular_tt',
            'after_decompose_reconstruct',
            model,
            tokenizer,
            baseline_target_weight,
            method_stats=method_stats,
        )
    )
    clean_memory_local(model, tokenizer)

    # dense-sparse TT
    for frac in OUTLIER_FRACTIONS:
        model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
        stats = decompose_target_layer_dense_sparse_inplace(model, tt_rank=TT_RANK, outlier_fraction=frac)
        reconstruct_target_layer_inplace(model)
        rows.append(stage_row('expA_no_lora', 'dense_sparse_tt', 'after_decompose_reconstruct', model, tokenizer, baseline_target_weight, outlier_fraction=frac, method_stats=stats))
        clean_memory_local(model, tokenizer)

    return pd.DataFrame(rows)

In [ ]:
cleanup_memory()

In [ ]:
expA_df = experiment_no_lora_methods()
display(expA_df)


expA_display_cols = [
    'experiment', 'method', 'outlier_fraction', 'stage', 'wikitext2_ppl',
    'superweight_rel_error_mean', 'target_layer_fro_rel_error',
    'kept_outlier_count', 'sparse_tt_max_rank', 'combined_tt_max_rank',
    'method_compression_ratio'
]
display(expA_df[[c for c in expA_display_cols if c in expA_df.columns]])

In [ ]:
plot_df = expA_df.copy()
plot_df['label'] = plot_df.apply(lambda r: 'baseline' if r['method'] == 'baseline' else (f"regular_tt_r{TT_RANK}" if r['method'] == 'regular_tt' else f"dense_sparse_{r['outlier_fraction']:.0e}"), axis=1)

plt.figure(figsize=(8.5, 4.8))
plt.plot(plot_df['label'], plot_df['wikitext2_ppl'], marker='o')
plt.xticks(rotation=30, ha='right')
plt.ylabel('WikiText-2 PPL')
plt.title('Baseline vs regular TT vs dense-sparse TT')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8.5, 4.8))
plt.plot(plot_df['label'], plot_df['superweight_rel_error_mean'], marker='o')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Superweight relative error')
plt.title('Superweight preservation across methods')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
cleanup_memory()

## Experiment B: LoRA after decompose → reconstruct

For each method:

1. baseline
2. decompose → reconstruct
3. LoRA train on the reconstructed dense layer
4. merge LoRA
5. decompose → reconstruct again with the same method

In [ ]:
def run_lora_pipeline(*, method: str, outlier_fraction=None, seed: int = 0):
    rows = []
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    baseline_target_weight = get_effective_target_weight(model)

    rows.append(stage_row('expB_lora', method, 'baseline', model, tokenizer, baseline_target_weight, outlier_fraction=outlier_fraction))

    if method == 'regular_tt':
        method_stats = decompose_target_layer_regular_inplace(model, tt_rank=TT_RANK)
    elif method == 'dense_sparse_tt':
        method_stats = decompose_target_layer_dense_sparse_inplace(model, tt_rank=TT_RANK, outlier_fraction=outlier_fraction)
    else:
        raise ValueError(f'Unknown method: {method}')

    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('expB_lora', method, 'after_first_decompose_reconstruct', model, tokenizer, baseline_target_weight, outlier_fraction=outlier_fraction, method_stats=method_stats))

    attach_lora_to_target_module(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
    print(f'[{method}, frac={outlier_fraction}] trainable parameters:', count_trainable_parameters(model))
    losses = run_lora_training(
        model,
        tokenizer,
        max_steps=MAX_STEPS,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        batch_size=TRAIN_BATCH_SIZE,
        seq_len=TRAIN_SEQ_LEN,
        num_sequences=NUM_TRAIN_SEQUENCES,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        seed=seed,
    )
    rows.append(stage_row('expB_lora', method, 'after_lora_training', model, tokenizer, baseline_target_weight, outlier_fraction=outlier_fraction))

    merge_target_lora_to_dense_inplace(model)
    rows.append(stage_row('expB_lora', method, 'after_lora_merge', model, tokenizer, baseline_target_weight, outlier_fraction=outlier_fraction))

    if method == 'regular_tt':
        method_stats2 = decompose_target_layer_regular_inplace(model, tt_rank=TT_RANK)
    else:
        method_stats2 = decompose_target_layer_dense_sparse_inplace(model, tt_rank=TT_RANK, outlier_fraction=outlier_fraction)
    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('expB_lora', method, 'after_second_decompose_reconstruct', model, tokenizer, baseline_target_weight, outlier_fraction=outlier_fraction, method_stats=method_stats2))

    out = pd.DataFrame(rows)
    clean_memory_local(model, tokenizer)
    return out, losses

In [ ]:
lora_tables = []
loss_histories = {}

regular_df, regular_losses = run_lora_pipeline(method='regular_tt', seed=0)
lora_tables.append(regular_df)
loss_histories['regular_tt'] = regular_losses

display(regular_df)

for idx, frac in enumerate(OUTLIER_FRACTIONS):
    ds_df, ds_losses = run_lora_pipeline(method='dense_sparse_tt', outlier_fraction=frac, seed=idx + 1)
    lora_tables.append(ds_df)
    loss_histories[f'dense_sparse_{frac:.0e}'] = ds_losses
    display(ds_df)

expB_df = pd.concat(lora_tables, ignore_index=True)

In [ ]:
display(expB_df)
expB_df.to_json(OUTPUT_JSON, orient='records', indent=2)
print('Saved:', OUTPUT_JSON)


expB_display_cols = [
    'experiment', 'method', 'outlier_fraction', 'stage', 'wikitext2_ppl',
    'superweight_rel_error_mean', 'target_layer_fro_rel_error',
    'outlier_count_exact', 'sparse_tt_max_rank', 'combined_tt_max_rank',
    'method_compression_ratio'
]
display(expB_df[[c for c in expB_display_cols if c in expB_df.columns]])

In [ ]:
for name, hist in loss_histories.items():
    plt.figure(figsize=(7, 4.2))
    plt.plot(hist['step'], hist['loss'], marker='o')
    plt.xlabel('Optimization step')
    plt.ylabel('Training loss')
    plt.title(f'LoRA loss vs step: {name}')
    plt.grid(True)
    plt.show()

In [ ]:
for metric in ['wikitext2_ppl', 'superweight_rel_error_mean']:
    plt.figure(figsize=(8.5, 4.8))
    for (method, frac), group in expB_df.groupby(['method', 'outlier_fraction'], dropna=False):
        label = 'regular_tt' if method == 'regular_tt' else f'dense_sparse_{frac:.0e}'
        plt.plot(group['stage'], group[metric], marker='o', label=label)
    plt.xticks(rotation=35, ha='right')
    plt.ylabel(metric)
    plt.title(f'{metric} across LoRA pipeline stages')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()